# mc-cuda-tinygrad -- loop smoke test

Purpose: prove the create -> push -> status -> output loop end to end before
spending any real accelerator time. This notebook runs for seconds.

It writes `smoke.txt` into `/kaggle/working/`, which is what comes back in the
output zip that `kaggle kernels output` downloads.

In [ ]:
import os, platform, subprocess, sys

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
lines = []

def emit(s):
    print(s)
    lines.append(str(s))

def sh(cmd):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
        return (p.stdout + p.stderr).strip()
    except Exception as e:
        return f"<{type(e).__name__}: {e}>"

emit(f"python      {sys.version.split()[0]}  ({sys.executable})")
emit(f"platform    {platform.platform()}")
emit(f"cwd         {os.getcwd()}")
emit(f"kaggle_input{os.path.isdir('/kaggle/input')}  kaggle_working={os.path.isdir(OUT)}")

for k in sorted(os.environ):
    if k.startswith(("KAGGLE", "CUDA", "NVIDIA", "TPU", "PJRT", "XLA", "TF_", "LD_LIBRARY")):
        emit(f"env {k}={os.environ[k][:120]}")

with open(os.path.join(OUT, "smoke.txt"), "w") as f:
    f.write("\n".join(lines) + "\n")
print(f"wrote {len(lines)} lines to {OUT}/smoke.txt")

In [ ]:
import os, subprocess, sys

OUT = "/kaggle/working"
report = []

def sh(cmd):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=120)
        return (p.stdout + p.stderr).strip()
    except Exception as e:
        return f"<{type(e).__name__}: {e}>"

def emit(s):
    print(s)
    report.append(str(s))

emit("=== nvidia-smi ===")
emit(sh("nvidia-smi"))
emit(sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader"))
emit("=== disk / cpu / ram ===")
emit(sh("df -h / | tail -2"))
emit(sh("nproc"))
emit(sh("free -g | head -2"))
emit("=== preinstalled ===")
for mod in ("torch", "tensorflow", "jax", "triton", "tinygrad", "numpy", "pandas"):
    out = sh(f"{sys.executable} -c \"import {mod}; print(getattr({mod}, '__version__', '?'))\" 2>&1 | tail -1")
    emit(f"  {mod:<12} {out[:100]}")
emit("=== git / clang ===")
emit("git  " + sh("git --version"))
emit("clang " + sh("clang --version | head -1"))
emit("gcc  " + sh("gcc --version | head -1"))
with open(os.path.join(OUT, "smoke-device.txt"), "w") as f:
    f.write("\n".join(report) + "\n")
print(f"wrote {len(report)} lines to {OUT}/smoke-device.txt")